# 🌡️ 01a — Pipeline météo

Construit `dim_meteo.parquet` (dept × annee_mois). Lancer `00_config_commun.ipynb` avant.

Source : Météo France, données climatologiques mensuelles (MENSQ).
https://www.data.gouv.fr/datasets/donnees-climatologiques-de-base-mensuelles
Fichiers dans `data/raw/MeteoFrance/MENSQ_XX_{previous-1950-2024,latest-2025-2026}.csv.gz`,
un par département (`XX`), Corse regroupée sous le code `20` à séparer en 2A/2B.

On utilisait SYNOP avant (42 stations, 41 départements couverts). MENSQ est
plus dense et déjà agrégé au mois par station, donc plus besoin de ré-agréger
des relevés horaires, et on couvre les 96 départements une fois la Corse
séparée en 2A/2B.

Correspondance des colonnes :

| `dim_meteo` | MENSQ | Description des variables |
|---|---|---|
| `temp_moy` | `TM` | Température moyenne mensuelle (°C) |
| `temp_min` | `TN` | Moyenne des températures minimales (°C) |
| `temp_max` | `TX` | Moyenne des températures maximales (°C) |
| `humidite_moy` | `UMM` | Humidité relative moyenne (%) |
| `vent_moy` | `FFM` | Vitesse moyenne du vent à 10 m (m/s) |
| `precip_total` | `RR` | Cumul mensuel des précipitations (mm) |

In [1]:
# Préambule : on se place dans le répertoire racine du projet et on ajoute le répertoire courant au PYTHONPATH pour pouvoir importer src/config.py

# pour recharger automatiquement les modules modifiés （src config surtout） sans redémarrer le kernel
%load_ext autoreload 
%autoreload 2

import os
import sys
from pathlib import Path
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, str(Path.cwd()))

import numpy as np
import pandas as pd
from src.config import RAW_DIR, TABLES_DIR, ANNEE_DEBUT, ANNEE_FIN, DEPTS,  DEPT_NOM_TO_CODE
from src.validation import valider_dim_table

print(
    f"Config chargée depuis src/config.py : {len(DEPTS)} départements | {ANNEE_DEBUT}–{ANNEE_FIN}")
print(f"RAW_DIR    = {RAW_DIR}")
print(f"TABLES_DIR = {TABLES_DIR}")

Config chargée depuis src/config.py : 96 départements | 2020–2025
RAW_DIR    = /Users/siranh/Documents/Data Scientest/projet_liora/data/raw
TABLES_DIR = /Users/siranh/Documents/Data Scientest/projet_liora/data/processed


In [2]:
# Vérifier la structure d'un fichier MENSQ (1 fichier = 1 département)
df_check = pd.read_csv(
    "data/raw/MeteoFrance/MENSQ_01_previous-1950-2024.csv.gz",
    sep=";", compression="gzip", nrows=5)
print("=== MENSQ (dept 01) ===")
print(df_check.columns.tolist())
display(df_check[["NUM_POSTE", "NOM_USUEL", "LAT", "LON", "AAAAMM",
                   "TM", "TX", "TN", "UMM", "FFM", "RR"]].head())

=== MENSQ (dept 01) ===
['NUM_POSTE', 'NOM_USUEL', 'LAT', 'LON', 'ALTI', 'AAAAMM', 'RR', 'QRR', 'NBRR', 'RR_ME', 'RRAB', 'QRRAB', 'RRABDAT', 'NBJRR1', 'NBJRR5', 'NBJRR10', 'NBJRR30', 'NBJRR50', 'NBJRR100', 'PMERM', 'QPMERM', 'NBPMERM', 'PMERMINAB', 'QPMERMINAB', 'PMERMINABDAT', 'TX', 'QTX', 'NBTX', 'TX_ME', 'TXAB', 'QTXAB', 'TXDAT', 'TXMIN', 'QTXMIN', 'TXMINDAT', 'NBJTX0', 'NBJTX25', 'NBJTX30', 'NBJTX35', 'NBJTXI20', 'NBJTXI27', 'NBJTXS32', 'TN', 'QTN', 'NBTN', 'TN_ME', 'TNAB', 'QTNAB', 'TNDAT', 'TNMAX', 'QTNMAX', 'TNMAXDAT', 'NBJTN5', 'NBJTN10', 'NBJTNI10', 'NBJTNI15', 'NBJTNI20', 'NBJTNS20', 'NBJTNS25', 'NBJGELEE', 'TAMPLIM', 'QTAMPLIM', 'TAMPLIAB', 'QTAMPLIAB', 'TAMPLIABDAT', 'NBTAMPLI', 'TM', 'QTM', 'NBTM', 'TMM', 'QTMM', 'NBTMM', 'NBJTMS24', 'TMMIN', 'QTMMIN', 'TMMINDAT', 'TMMAX', 'QTMMAX', 'TMMAXDAT', 'UNAB', 'QUNAB', 'UNABDAT', 'NBUN', 'UXAB', 'QUXAB', 'UXABDAT', 'NBUX', 'UMM', 'QUMM', 'NBUM', 'TSVM', 'QTSVM', 'NBTSVM', 'ETP', 'QETP', 'FXIAB', 'QFXIAB', 'DXIAB', 'QDXIAB', 'FXIDA

,NUM_POSTE,NOM_USUEL,LAT,LON,AAAAMM,TM,TX,TN,UMM,FFM,RR
0,1010001,ANGLEFORT,45.913667,5.809833,195001,NaN,NaN,NaN,NaN,NaN,49.5
1,1010001,ANGLEFORT,45.913667,5.809833,195002,NaN,NaN,NaN,NaN,NaN,237.6
2,1010001,ANGLEFORT,45.913667,5.809833,195003,NaN,NaN,NaN,NaN,NaN,23.8
3,1010001,ANGLEFORT,45.913667,5.809833,195005,NaN,NaN,NaN,NaN,NaN,83.6
4,1010001,ANGLEFORT,45.913667,5.809833,195006,NaN,NaN,NaN,NaN,NaN,75.3


In [3]:
import reverse_geocoder as rg
from src.config import DEPT_NOM_TO_CODE


def split_corse_2a_2b(df_20: pd.DataFrame) -> dict:
    """
    Le fichier MENSQ_20_* regroupe toutes les stations de Corse sous un seul code département "20", sans distinction 2A (Corse-du-Sud) / 2B (Haute-Corse).

    On identifie le bon département pour chaque station via un reverse geocoding sur sa latitude/longitude.

    Retourne : dict { NUM_POSTE (int) → "2A" ou "2B" }
    """
    stations = df_20[["NUM_POSTE", "LAT", "LON"]].drop_duplicates().copy()

    coords = list(zip(stations["LAT"], stations["LON"]))
    resultats = rg.search(coords)
    dept_nom = [r["admin2"] for r in resultats]

    # Nom département → code INSEE (cf. src/config.py) ; seuls 2A/2B nous
    # intéressent ici, le reste renverra None et sera filtré par le caller.
    stations["dept"] = [DEPT_NOM_TO_CODE.get(n) for n in dept_nom]

    non_resolues = stations["dept"].isna().sum()
    if non_resolues:
        print(f"  ⚠️  {non_resolues} stations corses non résolues (ignorées)")
    else:
        print(f"  ✅ {len(stations)} stations corses réparties en 2A/2B")

    return dict(zip(stations["NUM_POSTE"], stations["dept"]))

La cible (urgences) est mensuelle et on veut capter la saisonnalité fine
(allergie au printemps, bronchiolite en hiver). MENSQ étant déjà mensuel par
station, il reste juste à moyenner les stations d'un même département sur
le même mois.

In [4]:
def build_dim_meteo() -> pd.DataFrame:
    """
    Construit la table dim_meteo à partir des fichiers mensuels MENSQ de
    Météo France (un fichier par département, déjà agrégé au mois par station).

    Etapes:
    ───────────
    1. Charger, pour chaque département 01-95 (dont "20" pour la Corse),
       les fichiers "previous-1950-2024" et "latest-2025-2026"
    2. Filtrer sur la période ANNEE_DEBUT-ANNEE_FIN
    3. Cas particulier "20" : séparer les stations en 2A/2B (reverse geocoding,
       cf. split_corse_2a_2b ci-dessus)
    4. Renommer les champs MENSQ vers les noms de colonnes du schéma existant
    5. Agréger les stations d'un même département/mois → moyenne
    6. Sauvegarder en parquet

    VARIABLES PRODUITES : 
    - dept : code département (str)
    - annee_mois : "YYYY-MM" (str)
    - temp_moy : température moyenne (°C)
    - temp_max : température maximale (°C)
    - temp_min : température minimale (°C)
    - humidite_moy : humidité moyenne (%)
    - vent_moy : vitesse moyenne du vent (km/h)
    - precip_total : précipitations totales (mm)   

    CLÉ PRIMAIRE : dept × annee_mois
    """
    meteo_dir = RAW_DIR / "MeteoFrance"

    cols_utiles = ["NUM_POSTE", "LAT", "LON", "AAAAMM",
                   "TM", "TX", "TN", "UMM", "FFM", "RR"]
    RENAME_MAP = {
        "TM": "temp_moy", "TX": "temp_max", "TN": "temp_min",
        "UMM": "humidite_moy", "FFM": "vent_moy", "RR": "precip_total",
    }
    periodes = ["previous-1950-2024", "latest-2025-2026"]

    # ── ÉTAPE 1 : Charger chaque département (01-95, incl. "20" pour Corse) ──
    dfs = []
    df_20 = None  # traité à part le temps de la scission 2A/2B

    for code in [f"{i:02d}" for i in range(1, 96)]:
        frames = []
        for periode in periodes:
            fpath = meteo_dir / f"MENSQ_{code}_{periode}.csv.gz"
            if fpath.exists():
                df_periode = pd.read_csv(
                    fpath, sep=";", compression="gzip",
                    usecols=lambda c: c in cols_utiles, low_memory=False
                )
                frames.append(df_periode)

        if len(frames) == 0:
            print(f"  ⚠️  Aucun fichier pour le département {code}")
            continue

        df_code = pd.concat(frames, ignore_index=True)
        df_code["annee"] = df_code["AAAAMM"] // 100
        df_code = df_code[(df_code["annee"] >= ANNEE_DEBUT) & (df_code["annee"] <= ANNEE_FIN)]

        if code == "20":
            df_20 = df_code
        else:
            df_code = df_code.copy()
            df_code["dept"] = code
            dfs.append(df_code)

    # ── ÉTAPE 2 : Cas particulier de la Corse (scission 2A/2B) ───────────────
    if df_20 is not None and not df_20.empty:
        station_to_dept_corse = split_corse_2a_2b(df_20)
        df_20 = df_20.copy()
        df_20["dept"] = df_20["NUM_POSTE"].map(station_to_dept_corse)
        df_20 = df_20.dropna(subset=["dept"])
        dfs.append(df_20)

    if not dfs:
        print("❌ Aucun fichier MENSQ trouvé dans data/raw/MeteoFrance/")
        return pd.DataFrame()

    df = pd.concat(dfs, ignore_index=True)
    print(f"  Total brut {ANNEE_DEBUT}-{ANNEE_FIN} (tous départements) : {len(df):,} lignes")

    # ── ÉTAPE 3 : annee_mois = clé de jointure avec les autres tables ────────
    aaaamm_str = df["AAAAMM"].astype(str)
    df["annee_mois"] = aaaamm_str.str[:4] + "-" + aaaamm_str.str[4:6]

    # ── ÉTAPE 4 : Conversion numérique + renommage vers le schéma existant ──
    for col in RENAME_MAP:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df = df.rename(columns=RENAME_MAP)

    # ── ÉTAPE 5 : Agrégation des stations d'un même dept/mois → moyenne ─────
    # (chaque valeur MENSQ est déjà une moyenne mensuelle PAR STATION ;
    # ici on moyenne simplement entre les stations d'un même département)
    cols_sortie = list(RENAME_MAP.values())
    df_agg = (
        df.groupby(["dept", "annee_mois"])[cols_sortie]
        .mean()
        .round(2)
        .reset_index()
    )

    print(f"  ✅ dim_meteo : {df_agg.shape[0]:,} lignes × {df_agg.shape[1]} colonnes")
    print(f"  Période      : {df_agg['annee_mois'].min()} → {df_agg['annee_mois'].max()}")
    print(f"  Départements : {df_agg['dept'].nunique()} / {len(DEPTS)} couverts")
    return df_agg


# ══════════════════════════════════════════════════════════════════════════════
# EXÉCUTION
# ══════════════════════════════════════════════════════════════════════════════

print("Construction de dim_meteo...")
dim_meteo = build_dim_meteo()

if not dim_meteo.empty:
    valider_dim_table(dim_meteo, "dim_meteo")
    dim_meteo.to_parquet(TABLES_DIR / "dim_meteo.parquet", index=False)
    print(f"\n Sauvegardé → data/processed/dim_meteo.parquet")
    display(dim_meteo.head(10))

Construction de dim_meteo...
Loading formatted geocoded file...
  ✅ 81 stations corses réparties en 2A/2B
  Total brut 2020-2025 (tous départements) : 158,713 lignes
  ✅ dim_meteo : 6,912 lignes × 8 colonnes
  Période      : 2020-01 → 2025-12
  Départements : 96 / 96 couverts
── Validation de dim_meteo ──
  ✅ Tous les codes dept sont valides (96 départements)
  ✅ Aucun doublon sur la clé ['dept', 'annee_mois']
  Taux de valeurs manquantes :
    humidite_moy                     3.1%
    vent_moy                         3.1%
  ✅ OK — prêt pour la fusion (dim_meteo)


 Sauvegardé → data/processed/dim_meteo.parquet


,dept,annee_mois,temp_moy,temp_max,temp_min,humidite_moy,vent_moy,precip_total
0,01,2020-01,4.12,8.25,-0.02,84.00,1.82,60.11
1,01,2020-02,6.92,11.62,2.22,76.71,2.45,106.67
2,01,2020-03,7.54,13.10,1.98,69.71,2.22,94.82
3,01,2020-04,13.10,20.39,5.74,61.43,1.87,56.62
4,01,2020-05,15.24,21.60,8.88,69.00,2.17,96.62
5,01,2020-06,17.62,23.27,11.98,73.71,1.87,107.39
6,01,2020-07,21.26,28.72,13.84,59.86,2.13,23.59
7,01,2020-08,21.62,28.79,14.46,63.29,1.92,59.68
8,01,2020-09,17.68,24.14,11.23,72.00,1.72,93.63
9,01,2020-10,10.74,14.65,6.82,83.29,1.88,220.04


In [5]:
# Vérification de couverture : plus aucun département manquant ?
depts_manquants = sorted(set(DEPTS) - set(dim_meteo["dept"].unique()))
print(f"Départements sans aucune donnée météo : {depts_manquants or 'aucun ✅'}")

doublons = dim_meteo.duplicated(subset=["dept", "annee_mois"]).sum()
print(f"Doublons (dept, annee_mois) : {doublons}")

print("\nTaux de valeurs manquantes par colonne :")
print((dim_meteo.isna().mean() * 100).round(1))

Départements sans aucune donnée météo : aucun ✅
Doublons (dept, annee_mois) : 0

Taux de valeurs manquantes par colonne :
dept            0.0
annee_mois      0.0
temp_moy        0.0
temp_max        0.0
temp_min        0.0
humidite_moy    3.1
vent_moy        3.1
precip_total    0.0
dtype: float64
